# AI-Driven Smart Irrigation — Ireland (v2, leakage-free)

**What changed from v1 and why**

The first version predicted *today's* net irrigation requirement (NIR) while
`etc_mm` and `pe_mm` were in the feature list. Since `NIR = max(0, ETc - Pe)`,
the model was handed the two numbers the answer is computed from, so it scored
`R2 = 1.000`. That is target leakage, not performance.

**v2 predicts tomorrow's NIR (t+1) from information available today (t).**
Everything known on day t is allowed — today's weather, today's ET0/ETc/Pe/NIR,
lags and rolling windows. Nothing from t+1 is used. Tomorrow's weather is
genuinely unknown, so the model has something real to learn.

Every model is also compared against two baselines:

- **persistence** — "tomorrow looks like today"
- **climatology** — the day-of-year county average from the training years

A model only counts if it beats both.

**How to run:** open in Colab and choose Runtime > Run all. The dataset
downloads itself from the public repo, so nothing needs uploading. The last
cell prints a summary block to copy back.

In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib joblib openpyxl xgboost shap

In [ ]:
import os, math, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

# ---- config -----------------------------------------------------------
DATA_URL = ("https://raw.githubusercontent.com/Vignesh22-hub/smart-irrigation-ireland/"
            "main/Final%20Thesis/Data%20Sets/Integrated_Irish_Weather_2010_2024.xlsx")
DATA_PATH   = "Integrated_Irish_Weather_2010_2024.xlsx"
if not os.path.exists(DATA_PATH):            # pulled from the public repo
    import urllib.request
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print("downloaded", DATA_PATH, os.path.getsize(DATA_PATH), "bytes")
TRAIN_END   = "2019-12-31"      # train:  2010 - 2019
VALID_END   = "2022-12-31"      # valid:  2020 - 2022   test: 2023 - 2024
NIR_THRESHOLD_MM = 1.0          # irrigate if predicted NIR >= this
OUT_DIR     = "outputs"
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs("results", exist_ok=True)
os.makedirs("models", exist_ok=True)
print("config ready")

## 1. Agronomy and feature engineering

FAO-56 chain: `ET0 -> ETc = ET0 * Kc -> Pe -> NIR = max(0, ETc - Pe)`.
NIR is the **label**. The features are what a farmer would actually have on the
morning of day t.

In [ ]:
import math
import numpy as np
import pandas as pd

DEFAULT_KC = {1: 0.9, 2: 0.9, 3: 0.95, 4: 1.0, 5: 1.05, 6: 1.1,
              7: 1.1, 8: 1.05, 9: 1.0, 10: 0.95, 11: 0.92, 12: 0.9}

RENAME_MAP = {
    "date_time": "date", "Date": "date", "DATE": "date",
    "Temperature (°C)": "tmean_c", "temperature": "tmean_c", "tmean": "tmean_c",
    "tmin": "tmin_c", "tmax": "tmax_c",
    "Rainfall (mm)": "rain_mm", "rain": "rain_mm", "rainfall": "rain_mm", "precip_mm": "rain_mm",
    "Humidity (%)": "rh_pct", "humidity": "rh_pct", "Humidity": "rh_pct",
    "county_name": "county", "region": "county",
    "wind": "wind_ms", "Wind Speed (m/s)": "wind_ms",
    "solar": "solar_mj_m2_day",
}


# ---------------------------------------------------------------- agronomy
def ra_mj(latitude: float, doy: int) -> float:
    """Extraterrestrial radiation (MJ m-2 day-1), FAO-56 eq. 21."""
    phi = math.radians(latitude)
    dr = 1 + 0.033 * math.cos(2 * math.pi * doy / 365)
    delta = 0.409 * math.sin(2 * math.pi * doy / 365 - 1.39)
    x = -math.tan(phi) * math.tan(delta)
    x = max(-1.0, min(1.0, x))
    ws = math.acos(x)
    gsc = 0.0820
    return max((24 * 60 / math.pi) * gsc * dr *
               (ws * math.sin(phi) * math.sin(delta) +
                math.cos(phi) * math.cos(delta) * math.sin(ws)), 0.0)


def et0_hargreaves(tmean, ra, tmax=None, tmin=None):
    """Hargreaves-Samani ET0 (mm/day). Uses the full form when tmax/tmin exist."""
    ra_mm = ra / 2.45
    if tmax is not None and tmin is not None and not (pd.isna(tmax) or pd.isna(tmin)):
        td = max(tmax - tmin, 0.0)
        return max(0.0023 * ra_mm * (tmean + 17.8) * math.sqrt(td), 0.0)
    return max(0.0023 * ra_mm * (tmean + 17.8), 0.0)


def effective_rain(rain):
    """USDA-style effective rainfall heuristic."""
    r = max(float(rain), 0.0)
    return min(0.7 * r, r)


# --------------------------------------------------------------- features
def load_weather(path: str) -> pd.DataFrame:
    df = pd.read_excel(path) if str(path).endswith((".xlsx", ".xls")) else pd.read_csv(path)
    df = df.rename(columns={k: v for k, v in RENAME_MAP.items() if k in df.columns})
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)
    if "tmean_c" not in df and {"tmin_c", "tmax_c"}.issubset(df.columns):
        df["tmean_c"] = (df["tmin_c"] + df["tmax_c"]) / 2
    if "latitude" not in df:
        df["latitude"] = 53.5
    if "county" not in df:
        df["county"] = "Ireland"
    return df


def build_agronomy(df: pd.DataFrame) -> pd.DataFrame:
    """ET0 -> ETc -> Pe -> NIR. NIR is the *label*, derived, never a feature."""
    df = df.copy()
    df["month"] = df["date"].dt.month
    df["doy"] = df["date"].dt.dayofyear
    df["ra"] = [ra_mj(lat, doy) for lat, doy in zip(df["latitude"], df["doy"])]
    tmax = df["tmax_c"] if "tmax_c" in df else pd.Series([None] * len(df), index=df.index)
    tmin = df["tmin_c"] if "tmin_c" in df else pd.Series([None] * len(df), index=df.index)
    df["et0_mm"] = [et0_hargreaves(t, r, tx, tn)
                    for t, r, tx, tn in zip(df["tmean_c"], df["ra"], tmax, tmin)]
    df["kc"] = df["month"].map(DEFAULT_KC)
    df["etc_mm"] = df["et0_mm"] * df["kc"]
    df["pe_mm"] = df["rain_mm"].apply(effective_rain)
    df["nir_mm"] = (df["etc_mm"] - df["pe_mm"]).clip(lower=0)
    return df


LAG_DAYS = (1, 2, 3, 7)
ROLL_WINDOWS = (3, 7, 14, 30)


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """Only information available on day t. Target is day t+1."""
    parts = []
    for _, g in df.groupby("county", sort=False):
        g = g.sort_values("date").copy()

        for lag in LAG_DAYS:
            g[f"rain_lag_{lag}"] = g["rain_mm"].shift(lag)
            g[f"tmean_lag_{lag}"] = g["tmean_c"].shift(lag)
            g[f"nir_lag_{lag}"] = g["nir_mm"].shift(lag)
        for w in ROLL_WINDOWS:
            g[f"rain_sum_{w}d"] = g["rain_mm"].rolling(w, min_periods=1).sum()
            g[f"tmean_mean_{w}d"] = g["tmean_c"].rolling(w, min_periods=1).mean()
            g[f"et0_sum_{w}d"] = g["et0_mm"].rolling(w, min_periods=1).sum()
            g[f"nir_mean_{w}d"] = g["nir_mm"].rolling(w, min_periods=1).mean()

        dry = np.zeros(len(g), dtype=float)
        counter = 0.0
        for i, r in enumerate(g["rain_mm"].values):
            counter = 0.0 if r >= 1.0 else counter + 1.0
            dry[i] = counter
        g["days_since_rain"] = dry

        g["doy_sin"] = np.sin(2 * np.pi * g["doy"] / 365.25)
        g["doy_cos"] = np.cos(2 * np.pi * g["doy"] / 365.25)
        g["awc_mm"] = g["awc_mm"] if "awc_mm" in g else 100.0

        # TARGET: tomorrow's irrigation requirement
        g["nir_next_day"] = g["nir_mm"].shift(-1)
        parts.append(g)

    out = pd.concat(parts, axis=0).dropna(subset=["nir_next_day"])
    # the first few rows of each county have empty lag windows - drop them
    lag_cols = [c for c in out.columns if "_lag_" in c]
    out = out.dropna(subset=lag_cols).reset_index(drop=True)
    return out


def feature_columns(df: pd.DataFrame) -> list:
    """Every quantity KNOWN on day t, including today's weather and today's
    ET0/ETc/Pe/NIR. Those are not leakage here because the target is day t+1,
    whose weather is unknown. The only banned columns are the target itself and
    anything derived from t+1."""
    banned = {"nir_next_day", "ra"}
    allowed = []
    for c in df.columns:
        if c in banned or c in {"date", "county", "latitude"}:
            continue
        if df[c].dtype.kind in "if":
            allowed.append(c)
    return allowed


# ----------------------------------------------------------------- splits
def time_split(df, train_end="2019-12-31", valid_end="2022-12-31"):
    te, ve = pd.to_datetime(train_end), pd.to_datetime(valid_end)
    return (df[df["date"] <= te].copy(),
            df[(df["date"] > te) & (df["date"] <= ve)].copy(),
            df[df["date"] > ve].copy())


def regression_metrics(y, p):
    y, p = np.asarray(y, dtype=float), np.asarray(p, dtype=float)
    mae = float(np.mean(np.abs(y - p)))
    rmse = float(np.sqrt(np.mean((y - p) ** 2)))
    ss_res = float(np.sum((y - p) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else float("nan")
    return {"MAE": mae, "RMSE": rmse, "R2": r2}


def decision_metrics(y_true, y_pred, threshold=1.0):
    """Irrigate/skip decision quality at a given NIR threshold (mm)."""
    at = np.asarray(y_true) >= threshold
    ap = np.asarray(y_pred) >= threshold
    tp = int(np.sum(at & ap)); fp = int(np.sum(~at & ap))
    fn = int(np.sum(at & ~ap)); tn = int(np.sum(~at & ~ap))
    prec = tp / (tp + fp) if tp + fp else float("nan")
    rec = tp / (tp + fn) if tp + fn else float("nan")
    f1 = 2 * prec * rec / (prec + rec) if prec and rec and not np.isnan(prec) and not np.isnan(rec) else float("nan")
    acc = (tp + tn) / len(at) if len(at) else float("nan")
    water_saved = float(np.sum(np.asarray(y_true)[~at & ~ap]))  # correctly skipped demand
    return {"precision": prec, "recall": rec, "f1": f1, "accuracy": acc,
            "correct_skips": tn, "false_skips": fn, "water_saved_mm": water_saved}

## 2. Load the data and derive NIR

In [ ]:
raw = load_weather(DATA_PATH)
print("raw shape:", raw.shape)
print("columns:", list(raw.columns))
print("date range:", raw["date"].min().date(), "->", raw["date"].max().date())
print("counties:", raw["county"].nunique())

agro = build_agronomy(raw)
agro[["date", "county", "tmean_c", "rain_mm", "et0_mm", "etc_mm", "pe_mm", "nir_mm"]].head()

## 3. Build features and the t+1 target

In [ ]:
features = build_features(agro)
FEATURES = feature_columns(features)

print(f"rows: {len(features):,}   features: {len(FEATURES)}")
print("target: nir_next_day (NIR on day t+1)")
assert "nir_next_day" not in FEATURES, "target leaked into features"
FEATURES

## 4. Time-aware split

No shuffling. The model is only ever trained on the past.

In [ ]:
train_df, valid_df, test_df = time_split(features, TRAIN_END, VALID_END)
for name, d in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    if len(d):
        print(f"{name:5s} {d['date'].min().date()} -> {d['date'].max().date()}  ({len(d):,} rows)")

X_train, y_train = train_df[FEATURES], train_df["nir_next_day"].values
X_valid, y_valid = valid_df[FEATURES], valid_df["nir_next_day"].values
X_test,  y_test  = test_df[FEATURES],  test_df["nir_next_day"].values

## 5. Baselines first

If a model cannot beat "tomorrow looks like today", it is not worth shipping.

In [ ]:
results = {}

# persistence: tomorrow = today's NIR
results["Baseline: persistence"] = regression_metrics(y_test, test_df["nir_mm"].values)

# climatology: day-of-year county mean from the training years
clim = train_df.groupby(["county", train_df["date"].dt.dayofyear])["nir_next_day"].mean()
fallback = float(train_df["nir_next_day"].mean())
clim_pred = np.array([clim.get((c, d), fallback)
                      for c, d in zip(test_df["county"], test_df["date"].dt.dayofyear)])
results["Baseline: climatology"] = regression_metrics(y_test, clim_pred)

for k, v in results.items():
    print(f"{k:24s} MAE={v['MAE']:.3f}  RMSE={v['RMSE']:.3f}  R2={v['R2']:.3f}")

## 6. Models

Ridge as a linear reference, Random Forest, and XGBoost. Each is tuned on the
validation years only, then evaluated once on the held-out test years.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
import joblib

models = {
    "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "RandomForest": RandomForestRegressor(
        n_estimators=500, min_samples_leaf=3, random_state=42, n_jobs=-1),
}

try:
    from xgboost import XGBRegressor
    models["XGBoost"] = XGBRegressor(
        n_estimators=700, learning_rate=0.05, max_depth=6,
        subsample=0.9, colsample_bytree=0.9, random_state=42, n_jobs=-1)
except ImportError:
    from sklearn.ensemble import HistGradientBoostingRegressor
    models["HistGradientBoosting"] = HistGradientBoostingRegressor(random_state=42)
    print("xgboost unavailable - using HistGradientBoosting instead")

preds = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    p_valid = model.predict(X_valid) if len(valid_df) else None
    p_test  = model.predict(X_test)
    preds[name] = p_test
    results[name] = regression_metrics(y_test, p_test)
    joblib.dump(model, f"models/{name.lower()}_nextday_nir.pkl")
    line = f"{name:24s} TEST  MAE={results[name]['MAE']:.3f}  RMSE={results[name]['RMSE']:.3f}  R2={results[name]['R2']:.3f}"
    if p_valid is not None:
        mv = regression_metrics(y_valid, p_valid)
        line += f"   | VALID R2={mv['R2']:.3f}"
    print(line)

## 7. Results table

This is the table that goes in the README, the CV and LinkedIn.

In [ ]:
table = pd.DataFrame(results).T[["MAE", "RMSE", "R2"]].round(3)
table = table.sort_values("MAE")

best_model = [k for k in table.index if not k.startswith("Baseline")][0]
best_mae = table.loc[best_model, "MAE"]
base_mae = table.loc["Baseline: persistence", "MAE"]
improvement = 100 * (base_mae - best_mae) / base_mae

print(table.to_string())
print(f"\nBest model: {best_model}")
print(f"Improvement over persistence baseline: {improvement:.1f}% lower MAE")

with open("results/metrics.md", "w") as f:
    f.write("# Model results - next-day net irrigation requirement (mm)\n\n")
    f.write(f"Test period: {test_df['date'].min().date()} to {test_df['date'].max().date()}"
            f" ({len(test_df):,} county-days)\n\n")
    f.write(table.to_markdown())
    f.write(f"\n\nBest model: **{best_model}** - {improvement:.1f}% lower MAE than a persistence baseline.\n")
table

## 8. The decision that matters: irrigate or skip

Farmers do not consume an R2. They act on a yes/no. This measures the quality of
that decision and the water correctly saved by skipping.

In [ ]:
dm = decision_metrics(y_test, preds[best_model], NIR_THRESHOLD_MM)
for k, v in dm.items():
    print(f"{k:18s} {v:.3f}" if isinstance(v, float) else f"{k:18s} {v}")

base_dm = decision_metrics(y_test, test_df["nir_mm"].values, NIR_THRESHOLD_MM)
print(f"\n(persistence baseline F1: {base_dm['f1']:.3f})")

with open("results/metrics.md", "a") as f:
    f.write(f"\n## Irrigate / skip decision (threshold {NIR_THRESHOLD_MM} mm)\n\n")
    f.write(f"- precision: {dm['precision']:.3f}\n- recall: {dm['recall']:.3f}\n"
            f"- F1: {dm['f1']:.3f}\n- correct skips: {dm['correct_skips']:,}\n"
            f"- water correctly not applied: {dm['water_saved_mm']:.0f} mm across the test period\n")

## 9. What drives the prediction

In [ ]:
model = models[best_model]
importances = None
if hasattr(model, "feature_importances_"):
    importances = model.feature_importances_
elif hasattr(model, "steps") and hasattr(model.steps[-1][1], "coef_"):
    importances = np.abs(model.steps[-1][1].coef_)

if importances is not None:
    order = np.argsort(importances)[::-1][:20]
    names = np.array(FEATURES)[order]
    vals = importances[order]
    plt.figure(figsize=(8, 6))
    plt.barh(names[::-1], vals[::-1])
    plt.title(f"{best_model} - top 20 features (next-day NIR)")
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/figures/feature_importance.png", dpi=200)
    plt.show()

# predicted vs actual, one county, test period
county = test_df["county"].iloc[0]
mask = test_df["county"] == county
plt.figure(figsize=(12, 4))
plt.plot(test_df.loc[mask, "date"], y_test[mask.values], label="actual", linewidth=1)
plt.plot(test_df.loc[mask, "date"], preds[best_model][mask.values], label="predicted", linewidth=1)
plt.title(f"Next-day NIR - {county} ({best_model})")
plt.ylabel("mm"); plt.legend(); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/figures/pred_vs_actual_{county}.png", dpi=200)
plt.show()

## 10. Optional: SHAP explanations

Skip this if it is slow — the feature importance plot above is enough for the README.

In [ ]:
try:
    import shap
    sample = X_test.sample(min(500, len(X_test)), random_state=42)
    explainer = shap.Explainer(models[best_model])
    shap_values = explainer(sample)
    shap.summary_plot(shap_values, sample, show=False)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/figures/shap_summary.png", dpi=200, bbox_inches="tight")
    plt.show()
except Exception as e:
    print("SHAP skipped:", e)

## 11. Optional: LSTM

Only run this if you want to claim an LSTM. It uses the last 14 days of each
county's history as a sequence. If you skip it, remove LSTM from your README,
CV and LinkedIn.

In [ ]:
RUN_LSTM = False   # set True to train

if RUN_LSTM:
    try:
        from tensorflow.keras.models import Sequential
        from tensorflow.keras.layers import LSTM, Dense
        from sklearn.preprocessing import StandardScaler

        LOOKBACK = 14
        scaler = StandardScaler().fit(X_train)

        def to_sequences(df_part):
            Xs, ys = [], []
            for _, g in df_part.groupby("county", sort=False):
                g = g.sort_values("date")
                arr = scaler.transform(g[FEATURES])
                tgt = g["nir_next_day"].values
                for i in range(LOOKBACK, len(g)):
                    Xs.append(arr[i - LOOKBACK:i]); ys.append(tgt[i])
            return np.array(Xs), np.array(ys)

        Xtr_s, ytr_s = to_sequences(train_df)
        Xte_s, yte_s = to_sequences(test_df)

        lstm = Sequential([LSTM(64, input_shape=(LOOKBACK, len(FEATURES))), Dense(1)])
        lstm.compile(optimizer="adam", loss="mse")
        lstm.fit(Xtr_s, ytr_s, epochs=15, batch_size=64, verbose=1)

        p_lstm = lstm.predict(Xte_s).ravel()
        results["LSTM"] = regression_metrics(yte_s, p_lstm)
        print("LSTM", results["LSTM"])
        lstm.save("models/lstm_nextday_nir.h5")
    except Exception as e:
        print("LSTM skipped:", e)
else:
    print("LSTM not run - do not claim it until you do")

## 12. Sample recommendation output

In [ ]:
latest = features.sort_values(["county", "date"]).groupby("county").tail(1).copy()
latest_pred = models[best_model].predict(latest[FEATURES])

out = latest[["county", "date"]].copy()
out["predicted_nir_next_day_mm"] = latest_pred.round(2)
out["recommendation"] = np.where(latest_pred >= NIR_THRESHOLD_MM, "IRRIGATE", "SKIP")
out.to_csv("results/sample_predictions.csv", index=False)
out

## 13. Summary to copy back

In [ ]:
summary = f'''
=== SMART IRRIGATION v2 - RESULTS ===
Data       : {raw['date'].min().date()} to {raw['date'].max().date()}, {raw['county'].nunique()} counties, {len(features):,} county-days
Target     : net irrigation requirement on day t+1 (mm)
Split      : train <= {TRAIN_END} | valid <= {VALID_END} | test after that ({len(test_df):,} rows)
Features   : {len(FEATURES)} (all known on day t; nothing from t+1)

{table.to_string()}

Best model : {best_model}
vs persistence baseline : {improvement:.1f}% lower MAE

Decision quality at {NIR_THRESHOLD_MM} mm threshold:
  precision {dm['precision']:.3f} | recall {dm['recall']:.3f} | F1 {dm['f1']:.3f}
  correct skips {dm['correct_skips']:,} | water correctly not applied {dm['water_saved_mm']:.0f} mm
'''
print(summary)
with open("results/summary.txt", "w") as f:
    f.write(summary)